# Phase 7: Benchmarking — Tamil LLM vs Sarvam-1

**Goal**: Produce a reproducible benchmark table proving our model beats Sarvam-1 on Tamil.

## Sarvam-1 Tamil Scores (published baseline)

| Benchmark | Sarvam-1 Tamil | Our Target |
|---|---|---|
| MMLU | 43.79 | **> 50** |
| ARC-Challenge | 57.04 | **> 62** |
| BoolQ | 79.51 | **> 83** |
| TriviaQA | 89.48 | **> 91** |
| Flores en→ta (chrF++) | 44.02 | **> 49** |
| XQUAD F1 | 41.89 | **> 50** |

**Models evaluated**: Our checkpoints (CPT, SFT, DPO) + Sarvam-1 + Llama 3.1 8B base (sanity check)

**Hardware**: Local 8GB or free Colab tier (inference only)

In [ ]:
# ── Install lm-evaluation-harness ──────────────────────────────────────────────
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "lm-eval[api]",
    "transformers",
    "accelerate",
    "datasets",
    "sacrebleu",    # for chrF++ / FLORES translation metric
    "tabulate",     # for results table
])
print("Ready.")

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────

# Models to evaluate — add/remove as checkpoints become available
MODELS = {
    "llama_3.1_8b_base":  "meta-llama/Meta-Llama-3.1-8B",          # no CPT sanity check
    "sarvam_1":           "sarvamai/sarvam-1",                       # competitor baseline
    "our_cpt_v1":         "wickkiey/tamil-llama-3.1-8b-cpt-v1",     # Phase 1
    "our_sft_v1":         "wickkiey/tamil-llama-3.1-8b-sft-v1",     # Phase 4
    "our_dpo_v1":         "wickkiey/tamil-llama-3.1-8b-dpo-v1",     # Phase 5
}

# Run on subset first (faster), then full for publication
EVAL_SUBSET = True    # True = fast 1K-sample eval; False = full eval for publication
NUM_FEWSHOT = 5       # 5-shot for all tasks (matches Sarvam eval setup)
LIMIT       = 200 if EVAL_SUBSET else None

HF_TOKEN    = None
RESULTS_DIR = "./eval_results"

import os
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Models to eval : {list(MODELS.keys())}")
print(f"Subset mode    : {EVAL_SUBSET} (limit={LIMIT})")

In [ ]:
# ── Part 1: Run lm-evaluation-harness tasks ────────────────────────────────────
# Tasks that map to Sarvam-1 benchmark tasks in Tamil
#
# Note: Tamil-specific task configs may need to be added to lm-eval-harness.
# We use the available multilingual variants:
#   - mmlu: multilingual MMLU (includes Tamil via IndicMMLU)
#   - arc_easy, arc_challenge: AI2 Reasoning Challenge
#   - boolq: Boolean questions
#   - xquad_ta: cross-lingual QA (Tamil)

import subprocess
import json

TASKS = [
    "arc_challenge",
    "boolq",
    "xquad_ta",       # may need: pip install lm_eval[xquad]
]

all_results = {}

for model_key, model_path in MODELS.items():
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_key} ({model_path})")
    print(f"{'='*60}")

    output_path = os.path.join(RESULTS_DIR, f"{model_key}.json")

    cmd = [
        sys.executable, "-m", "lm_eval",
        "--model", "hf",
        "--model_args", f"pretrained={model_path},load_in_4bit=True,dtype=bfloat16",
        "--tasks", ",".join(TASKS),
        "--num_fewshot", str(NUM_FEWSHOT),
        "--output_path", output_path,
        "--log_samples",
    ]
    if LIMIT:
        cmd += ["--limit", str(LIMIT)]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode == 0:
        with open(output_path, "r") as f:
            all_results[model_key] = json.load(f)
        print(f"Done. Saved to {output_path}")
    else:
        print(f"ERROR for {model_key}:")
        print(result.stderr[-2000:])

In [ ]:
# ── Part 2: Perplexity on held-out Tamil Wikipedia ─────────────────────────────
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import math

# Load 10% held-out split as perplexity test set
wiki_ds = load_dataset("wickkiey/tamil-wikipedia-markdown", split="train")
test_size = max(500, len(wiki_ds) // 10)
test_ds   = wiki_ds.shuffle(seed=999).select(range(test_size))
test_texts = [x["text"][:512] for x in test_ds if len(x["text"]) > 50]

print(f"Perplexity test set: {len(test_texts):,} chunks")

def compute_perplexity(model_path, texts, batch_size=8):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(
        model_path, torch_dtype=torch.bfloat16, device_map="auto"
    )
    model.eval()

    total_loss, total_tokens = 0.0, 0

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = tokenizer(batch, return_tensors="pt", padding=True,
                          truncation=True, max_length=512).to(model.device)
        with torch.inference_mode():
            outputs = model(**enc, labels=enc["input_ids"])
        loss   = outputs.loss.item()
        tokens = enc["attention_mask"].sum().item()
        total_loss   += loss * tokens
        total_tokens += tokens

    ppl = math.exp(total_loss / total_tokens)
    del model
    torch.cuda.empty_cache()
    return ppl

ppl_results = {}
for model_key, model_path in MODELS.items():
    print(f"Computing perplexity for {model_key}...")
    try:
        ppl = compute_perplexity(model_path, test_texts)
        ppl_results[model_key] = ppl
        print(f"  {model_key}: PPL = {ppl:.2f}")
    except Exception as e:
        print(f"  {model_key}: ERROR — {e}")
        ppl_results[model_key] = None

In [ ]:
# ── Part 3: Build comparison table ────────────────────────────────────────────
from tabulate import tabulate

SARVAM_SCORES = {
    "ARC-Challenge": 57.04,
    "BoolQ":         79.51,
    "XQUAD F1":      41.89,
    "TriviaQA":      89.48,
    "Perplexity":    None,  # not published
}

def extract_score(results_dict, model_key, task_key):
    """Extract primary metric from lm-eval results JSON."""
    if model_key not in results_dict:
        return None
    results = results_dict[model_key].get("results", {})
    task_results = results.get(task_key, {})
    # Try common metric keys
    for metric in ["acc_norm,none", "acc,none", "f1,none", "exact_match,none"]:
        if metric in task_results:
            return round(task_results[metric] * 100, 2)
    return None

# Build table rows
table_data = []
headers    = ["Model", "ARC-Challenge", "BoolQ", "XQUAD F1", "Perplexity (↓ better)"]

# Sarvam-1 row
table_data.append(["Sarvam-1 (competitor)", 57.04, 79.51, 41.89, "—"])

for model_key in MODELS:
    row = [
        model_key,
        extract_score(all_results, model_key, "arc_challenge"),
        extract_score(all_results, model_key, "boolq"),
        extract_score(all_results, model_key, "xquad_ta"),
        f"{ppl_results.get(model_key, 'N/A'):.2f}" if ppl_results.get(model_key) else "N/A",
    ]
    table_data.append(row)

print("\n" + "="*80)
print("BENCHMARK RESULTS: Tamil LLM vs Sarvam-1")
print("="*80)
print(tabulate(table_data, headers=headers, tablefmt="github", floatfmt=".2f"))
print("\n(Higher is better for all metrics except Perplexity)")

In [ ]:
# ── Save results to JSON for README ───────────────────────────────────────────
import json

summary = {
    "evaluation_date": __import__("datetime").date.today().isoformat(),
    "sarvam_1_baseline": SARVAM_SCORES,
    "perplexity": ppl_results,
    "raw_results_dir": RESULTS_DIR,
    "table": [
        dict(zip(headers, row)) for row in table_data
    ]
}

with open(os.path.join(RESULTS_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"Results saved to {RESULTS_DIR}/summary.json")
print("\nCopy the table above into README.md to publish the comparison.")

## Interpreting Results

| CPT vs Base | SFT vs CPT | DPO vs SFT | Meaning |
|---|---|---|---|
| PPL improves (↓) | Task scores improve | Task scores improve further | Pipeline working correctly |
| PPL improves but tasks don't | — | — | CPT helped fluency; SFT needed |
| All scores improve vs Sarvam-1 | — | — | ✅ Ready to publish |

## Publishing
1. Copy the GitHub-flavored markdown table to `README.md`
2. Tag release: `git tag -a dpo-v1 -m "DPO v1 — beats Sarvam-1 on Tamil"`
3. Push model card with benchmark table to HuggingFace